# Module 4 - Week 6, Topic 4: Random Forests
## Live Demo Notebook

**Scenario:** MTN Nigeria wants to predict which subscribers will churn next month.
We compare a single Decision Tree vs a Random Forest on the same data -
demonstrating improved accuracy AND improved stability.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_theme(style='whitegrid', palette='muted')

df = pd.read_csv('mtn_churn.csv')
print('Shape:', df.shape)
print(f'Churn rate: {df["churned"].mean():.1%}')
df.head()

In [ ]:
FEATURES = ['months_active', 'monthly_spend', 'num_complaints', 'data_usage_gb', 'roaming_active']
TARGET   = 'churned'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Training: {len(X_train)} | Test: {len(X_test)}')

---
## Part 1 - Decision Tree vs Random Forest: Direct Comparison

In [ ]:
# Single Decision Tree
dt = DecisionTreeClassifier(max_depth=10, random_state=42)
dt.fit(X_train, y_train)
dt_pred  = dt.predict(X_test)
dt_acc   = accuracy_score(y_test, dt_pred)

# Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred  = rf.predict(X_test)
rf_acc   = accuracy_score(y_test, rf_pred)

print('=== Accuracy Comparison ===')
print(f'Decision Tree:  {dt_acc:.2%}')
print(f'Random Forest:  {rf_acc:.2%}')
print(f'Improvement:    +{(rf_acc - dt_acc)*100:.1f} percentage points')

In [ ]:
print('=== Decision Tree Classification Report ===')
print(classification_report(y_test, dt_pred, target_names=['Stayed', 'Churned']))

print('=== Random Forest Classification Report ===')
print(classification_report(y_test, rf_pred, target_names=['Stayed', 'Churned']))

---
## Part 2 - Stability Test: 5 Random Seeds

In [ ]:
# Train both models with 5 different random seeds - show stability difference
seeds = [0, 10, 42, 99, 200]
dt_accs, rf_accs = [], []

print(f'{"seed":>6} {"Decision Tree":>16} {"Random Forest":>15}')
print('-' * 42)

for seed in seeds:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)

    dt_s = DecisionTreeClassifier(max_depth=10, random_state=seed)
    dt_s.fit(X_tr, y_tr)
    da = accuracy_score(y_te, dt_s.predict(X_te))

    rf_s = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=seed, n_jobs=-1)
    rf_s.fit(X_tr, y_tr)
    ra = accuracy_score(y_te, rf_s.predict(X_te))

    dt_accs.append(da)
    rf_accs.append(ra)
    print(f'{seed:>6} {da:>15.2%} {ra:>15.2%}')

print()
print(f'Decision Tree variance across seeds: ±{np.std(dt_accs)*100:.1f}%')
print(f'Random Forest variance across seeds: ±{np.std(rf_accs)*100:.1f}%')
print()
print('Lower variance = more stable, more trustworthy in production.')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(seeds))
width = 0.35

ax.bar(x - width/2, [a*100 for a in dt_accs], width, label='Decision Tree',
       color='coral', edgecolor='white')
ax.bar(x + width/2, [a*100 for a in rf_accs], width, label='Random Forest',
       color='steelblue', edgecolor='white')

ax.set_xlabel('Random Seed')
ax.set_ylabel('Test Accuracy (%)')
ax.set_xticks(x)
ax.set_xticklabels(seeds)
ax.set_title('Stability Comparison: Decision Tree vs Random Forest\n(5 different random splits)',
             fontsize=12, fontweight='bold')
ax.legend()
ax.set_ylim(60, 105)
plt.tight_layout()
plt.show()

---
## Part 3 - Feature Importance: Tree vs Forest

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Decision Tree importance (single seed)
dt_imp = pd.Series(dt.feature_importances_, index=FEATURES).sort_values()
dt_imp.plot(kind='barh', ax=ax1, color='coral', edgecolor='white')
ax1.set_title('Decision Tree Feature Importance\n(unstable - varies by seed)',
              fontsize=11, fontweight='bold')
ax1.set_xlabel('Importance Score')

# Random Forest importance (averaged across 100 trees)
rf_imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()
rf_imp.plot(kind='barh', ax=ax2, color='steelblue', edgecolor='white')
ax2.set_title('Random Forest Feature Importance\n(averaged across 100 trees - stable)',
              fontsize=11, fontweight='bold')
ax2.set_xlabel('Importance Score')

plt.suptitle('MTN Churn Prediction - Feature Importance Comparison',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Random Forest insight:')
print('Most important feature for MTN churn:', rf_imp.index[-1])
print('Business action: investigate and address this factor for at-risk subscribers.')

---
## Part 4 - OOB Score: Free Validation

In [ ]:
# Train on ALL training data - use OOB as validation instead of test set
rf_oob = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    oob_score=True,   # enable OOB scoring
    random_state=42,
    n_jobs=-1
)
rf_oob.fit(X_train, y_train)

print('=== OOB Score ===')
print(f'OOB score:      {rf_oob.oob_score_:.2%}')
print(f'Test score:     {accuracy_score(y_test, rf_oob.predict(X_test)):.2%}')
print()
print('OOB score uses ~37% of each bootstrap sample as free validation.')
print('Close to test score = reliable generalisation estimate.')

---
## Part 5 - Effect of n_estimators

In [ ]:
# How does accuracy change as we add more trees?
n_trees_range = [1, 5, 10, 20, 50, 100, 150, 200, 300]
test_accs = []

for n in n_trees_range:
    rf_n = RandomForestClassifier(n_estimators=n, max_depth=10, random_state=42, n_jobs=-1)
    rf_n.fit(X_train, y_train)
    test_accs.append(accuracy_score(y_test, rf_n.predict(X_test)))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(n_trees_range, [a*100 for a in test_accs], marker='o', color='steelblue',
        linewidth=2, markersize=7)
ax.axvline(100, color='red', linestyle='--', linewidth=1.5, label='n=100 (typical default)')
ax.set_xlabel('Number of Trees (n_estimators)')
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Accuracy vs Number of Trees - Diminishing Returns After ~100',
             fontsize=12, fontweight='bold')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

print('Accuracy stabilises quickly - adding trees beyond 100 gives diminishing returns.')
print('But more trees never hurts (only takes more training time).')

---
## Summary

| What we demonstrated | Key insight |
|---|---|
| DT vs RF accuracy | Random Forest [improvement]% more accurate on this dataset |
| Stability test | RF variance ±[std]% vs DT variance ±[std]% - RF much more stable |
| Feature importance | RF importance is averaged across 100 trees - reliable for business decisions |
| OOB score | Free validation mechanism - reliable estimate without a separate test set |
| n_estimators effect | Accuracy stabilises around 100 trees - diminishing returns beyond |

**Next - Topic 5:** Model Evaluation Metrics. We've been reporting accuracy - but accuracy can be misleading. We learn precision, recall, F1, confusion matrix, ROC curve, and AUC.